# 🌧️ ANUGA Overland Flow Simulation with IMERG Rainfall Data

## Overview
This notebook simulates overland flow using real rainfall data from IMERG (Integrated Multi-satellitE Retrievals for GPM). The rainfall data is provided as time-series at discrete grid points defined by latitude/longitude coordinates.

## Key Features:
- **Real Rainfall Data**: Uses IMERG hourly precipitation data from 2022
- **Cell-Based Rainfall Distribution**: Each rainfall cell has constant value = average of 4 corner grid points
- **No Interpolation**: Preserves original low rainfall values without artificial smoothing
- **GPU Acceleration**: Leverages CUDA/CuPy for 5-10x speedup
- **UTM Coordinate Transformation**: Converts WGS84 lat/lon to UTM Zone 43N
- **Time-varying Rainfall**: Applies rainfall that changes temporally (hourly updates)

## Rainfall Method:
Given the coarse spatial resolution of IMERG data (~10-11 km grid spacing) and already low rainfall values (mm/hr), interpolation is not appropriate. Instead, this notebook uses a **cell-based approach**:
- Each domain triangle is assigned to a rainfall cell
- Cell value = average of 4 corner grid points
- Constant rainfall within each cell for 1 hour duration
- This preserves the original data characteristics without artificial smoothing

## Data Sources:
- **DEM**: `DEM_UTM_EPSG32643.tif` (UTM Zone 43N)
- **Rainfall**: `IMERG_hourly_2022_ROI.csv` (WGS84 lat/lon grid)
- **Domain**: Derived from bounding box

## Requirements:
- ANUGA with CUDA support
- CuPy (for GPU acceleration)
- GDAL, pandas, numpy, scipy, matplotlib
- NVIDIA GPU with CUDA

## 📦 Import Libraries and Check GPU

In [1]:
# Core libraries
import anuga
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from scipy.spatial import cKDTree
import os
import sys
import time
from pathlib import Path

# Geospatial libraries
from osgeo import gdal, osr
from shapely import wkt
from pyproj import Transformer, CRS

# Check for GPU support
GPU_AVAILABLE = False
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("CuPy found - GPU acceleration ENABLED")
    print(f"  GPU Device: {cp.cuda.Device()}")
    print(f"  CuPy version: {cp.__version__}")
    
    # Get GPU info
    mempool = cp.get_default_memory_pool()
    mem_info = cp.cuda.Device().mem_info
    print(f"  GPU Memory Available: {mem_info[0]/1e9:.2f} GB free / {mem_info[1]/1e9:.2f} GB total")
except ImportError:
    print(" CuPy not found - Running on CPU only")
    print("  To enable GPU: pip install cupy-cuda11x (or cupy-cuda12x)")

print(f"\nANUGA version: {anuga.__version__ if hasattr(anuga, '__version__') else 'Unknown'}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

CuPy found - GPU acceleration ENABLED
  GPU Device: <CUDA Device 0>
  CuPy version: 13.6.0
  GPU Memory Available: 3.84 GB free / 3.95 GB total

ANUGA version: 3.2.1rc17
NumPy version: 2.3.5
Pandas version: 2.3.3


## Configuration

### 📅 How to Specify Custom Date Range

You can now run the simulation for any date range in your rainfall data by modifying the configuration line below.

**Format:** DD-MM-YYYY (e.g., "15-01-2022" for January 15, 2022)

**Examples:**

```python
# Run simulation from January 1 to January 10, 2022
config = RainfallSimulationConfig(start_date_str="01-01-2022", end_date_str="10-01-2022")

# Run simulation for the entire month of February 2022
config = RainfallSimulationConfig(start_date_str="01-02-2022", end_date_str="28-02-2022")

# Run simulation from June 15 to July 15, 2022
config = RainfallSimulationConfig(start_date_str="15-06-2022", end_date_str="15-07-2022")

# Use default (first 10 days of available data)
config = RainfallSimulationConfig(start_date_str=None, end_date_str=None)
```

**Notes:**
- Dates must be within the available rainfall data range
- If dates are outside the range, the code will automatically adjust to the nearest valid dates
- The end date will include all hours of that day (up to 23:59:59)

In [2]:
class RainfallSimulationConfig:
    """Configuration for rainfall-based overland flow simulation."""
    def __init__(self, start_date_str=None, end_date_str=None):
        # Input files
        self.dem_file = "filled_smoothed_topography_EPSG32643.tif"  # DEM in UTM Zone 43N
        self.rainfall_csv = "IMERG_hourly_2022_ROI.csv"  # IMERG data
        self.bounding_box_csv = "bounder.csv"  # Domain boundary
        
        # Coordinate system
        self.target_epsg = 32643  # UTM Zone 43N
        self.source_epsg = 4326   # WGS84 (lat/lon)
        
        # Domain parameters
        self.maximum_triangle_area = 30000  # m² (mesh resolution)
        self.buffer_distance = -5000.0  # meters (inward buffer)
        
        # Physical parameters
        self.friction_coefficient = 0.0175  # Manning's n
        self.minimum_storable_height = 1e-8 # meters
        
        # Simulation time parameters
        # User-specified dates in DD-MM-YYYY format
        self.start_date_str = start_date_str  # e.g., "01-01-2022"
        self.end_date_str = end_date_str      # e.g., "10-01-2022"
        
        # These will be set from data or user input
        self.start_date = None  # Will be set as datetime object
        self.end_date = None    # Will be set as datetime object
        
        self.output_interval_hours = 24.0  # Save results every hour
        
        # GPU and performance
        self.use_gpu = GPU_AVAILABLE
        self.save_checkpoints = True
        self.checkpoint_interval_hours = 24.0 * 15  # Checkpoint every 5 days
        
        # Output
        self.output_dir = 'rainfall_runoff_only'
        self.simulation_name = 'rainfall_only_simulation'
        
# USER INPUT: Specify your desired simulation period here (DD-MM-YYYY format)
# Example: config = RainfallSimulationConfig(start_date_str="01-01-2022", end_date_str="10-01-2022")
# Leave None to use first 10 days of available data
config = RainfallSimulationConfig(start_date_str="15-05-2022", end_date_str="15-09-2022")

print("Configuration loaded")
print(f"  DEM: {config.dem_file}")
print(f"  Rainfall data: {config.rainfall_csv}")
print(f"  Target EPSG: {config.target_epsg}")
print(f"  GPU enabled: {config.use_gpu}")
if config.start_date_str:
    print(f"  User-specified start date: {config.start_date_str}")
if config.end_date_str:
    print(f"  User-specified end date: {config.end_date_str}")
else:
    print(f"  Date range: Will use available data (default: first 10 days)")

Configuration loaded
  DEM: filled_smoothed_topography_EPSG32643.tif
  Rainfall data: IMERG_hourly_2022_ROI.csv
  Target EPSG: 32643
  GPU enabled: True
  User-specified start date: 15-05-2022
  User-specified end date: 15-09-2022


## 📊 Load and Process Rainfall Data

In [3]:
print("\n" + "="*70)
print("LOADING IMERG RAINFALL DATA")
print("="*70)

# Load rainfall CSV
print(f"\nReading {config.rainfall_csv}...")
rain_df = pd.read_csv(config.rainfall_csv)

print(f"aded {len(rain_df):,} records")
print(f"\nColumns: {list(rain_df.columns)}")
print(f"\nFirst few rows:")
print(rain_df.head())

# Parse time column (format: DD-MM-YYYY HH:MM)
print("\nParsing timestamps...")
rain_df['datetime'] = pd.to_datetime(rain_df['time'], format='%d-%m-%Y %H:%M')

# Get unique timestamps and spatial points
unique_times = sorted(rain_df['datetime'].unique())
unique_lats = sorted(rain_df['lat'].unique())
unique_lons = sorted(rain_df['long'].unique())

print(f"rsed {len(unique_times)} unique timestamps")
print(f"  Available time range: {unique_times[0]} to {unique_times[-1]}")
print(f"  Total duration: {(unique_times[-1] - unique_times[0]).days} days, {(unique_times[-1] - unique_times[0]).seconds//3600} hours")

print(f"\natial grid: {len(unique_lats)} latitudes × {len(unique_lons)} longitudes")
print(f"  Latitude range: {min(unique_lats):.3f}° to {max(unique_lats):.3f}°")
print(f"  Longitude range: {min(unique_lons):.3f}° to {max(unique_lons):.3f}°")

# Statistics
print(f"\necipitation statistics:")
print(f"  Min: {rain_df['precipitation'].min():.6f} mm/hr")
print(f"  Max: {rain_df['precipitation'].max():.6f} mm/hr")
print(f"  Mean: {rain_df['precipitation'].mean():.6f} mm/hr")
print(f"  Non-zero values: {(rain_df['precipitation'] > 0).sum():,} ({(rain_df['precipitation'] > 0).sum()/len(rain_df)*100:.1f}%)")

# Set simulation period based on user input or defaults
print(f"\n{'='*70}")
print("SETTING SIMULATION PERIOD")
print(f"{'='*70}")

if config.start_date_str:
    # Parse user-specified start date (DD-MM-YYYY format)
    config.start_date = pd.to_datetime(config.start_date_str, format='%d-%m-%Y')
    print(f"ing user-specified start date: {config.start_date}")
    
    # Validate start date is within available data
    if config.start_date < unique_times[0]:
        print(f"  ⚠ WARNING: Start date {config.start_date} is before available data starts {unique_times[0]}")
        print(f"  Using earliest available date: {unique_times[0]}")
        config.start_date = unique_times[0]
    elif config.start_date > unique_times[-1]:
        raise ValueError(f"Start date {config.start_date} is after available data ends {unique_times[-1]}")
else:
    # Use first available date
    config.start_date = unique_times[0]
    print(f"Using first available date: {config.start_date}")

if config.end_date_str:
    # Parse user-specified end date (DD-MM-YYYY format)
    config.end_date = pd.to_datetime(config.end_date_str, format='%d-%m-%Y')
    # Set to end of day (23:59:59) to include all hours of the end date
    config.end_date = config.end_date.replace(hour=23, minute=59, second=59)
    print(f"ing user-specified end date: {config.end_date}")
    
    # Validate end date is within available data
    if config.end_date > unique_times[-1]:
        print(f"  ⚠ WARNING: End date {config.end_date} is after available data ends {unique_times[-1]}")
        print(f"  Using latest available date: {unique_times[-1]}")
        config.end_date = unique_times[-1]
    elif config.end_date < config.start_date:
        raise ValueError(f"End date {config.end_date} is before start date {config.start_date}")
else:
    # Default: 10 days from start date (or end of available data, whichever is earlier)
    default_end = config.start_date + timedelta(days=10)
    config.end_date = min(default_end, unique_times[-1])
    print(f"ing default period (10 days): {config.end_date}")

print(f"\n{'='*70}")
print(f"FINAL SIMULATION PERIOD: {config.start_date} to {config.end_date}")
print(f"Duration: {(config.end_date - config.start_date).days} days, {(config.end_date - config.start_date).seconds//3600} hours")
print(f"{'='*70}")


LOADING IMERG RAINFALL DATA

Reading IMERG_hourly_2022_ROI.csv...
aded 960,960 records

Columns: ['time', 'lat', 'long', 'precipitation']

First few rows:
               time   lat  long  precipitation
0  01-01-2022 00:00  30.0  75.0         0.0555
1  01-01-2022 00:00  30.1  75.0         0.0580
2  01-01-2022 00:00  30.2  75.0         0.0615
3  01-01-2022 00:00  30.3  75.0         0.0650
4  01-01-2022 00:00  30.4  75.0         0.0675

Parsing timestamps...
rsed 8736 unique timestamps
  Available time range: 2022-01-01 00:00:00 to 2022-12-30 23:00:00
  Total duration: 363 days, 23 hours

atial grid: 10 latitudes × 11 longitudes
  Latitude range: 30.000° to 30.900°
  Longitude range: 75.000° to 76.000°

ecipitation statistics:
  Min: 0.000000 mm/hr
  Max: 33.924999 mm/hr
  Mean: 0.077613 mm/hr
  Non-zero values: 69,721 (7.3%)

SETTING SIMULATION PERIOD
ing user-specified start date: 2022-05-15 00:00:00
ing user-specified end date: 2022-09-15 23:59:59

FINAL SIMULATION PERIOD: 2022-05-15 

## 🗺️ Transform Rainfall Coordinates to UTM

In [4]:
print("\n" + "="*70)
print("COORDINATE TRANSFORMATION: WGS84 → UTM Zone 43N")
print("="*70)

# Create transformer from WGS84 to UTM Zone 43N
print(f"\nCreating coordinate transformer...")
print(f"  Source: EPSG:{config.source_epsg} (WGS84 lat/lon)")
print(f"  Target: EPSG:{config.target_epsg} (UTM Zone 43N)")

transformer = Transformer.from_crs(
    CRS.from_epsg(config.source_epsg),
    CRS.from_epsg(config.target_epsg),
    always_xy=True
)

# Transform coordinates
print("\nTransforming rainfall grid coordinates...")
rain_df['easting'], rain_df['northing'] = transformer.transform(
    rain_df['long'].values,
    rain_df['lat'].values
)

print(f"Coordinates transformed")
print(f"\nUTM coordinates:")
print(f"  Easting range: {rain_df['easting'].min():.2f} to {rain_df['easting'].max():.2f} m")
print(f"  Northing range: {rain_df['northing'].min():.2f} to {rain_df['northing'].max():.2f} m")

# Show example transformation
print(f"\nExample transformation:")
sample = rain_df.iloc[0]
print(f"  Lat/Lon: ({sample['lat']:.6f}°, {sample['long']:.6f}°)")
print(f"  UTM: ({sample['easting']:.2f} m E, {sample['northing']:.2f} m N)")


COORDINATE TRANSFORMATION: WGS84 → UTM Zone 43N

Creating coordinate transformer...
  Source: EPSG:4326 (WGS84 lat/lon)
  Target: EPSG:32643 (UTM Zone 43N)

Transforming rainfall grid coordinates...

Transforming rainfall grid coordinates...
Coordinates transformed

UTM coordinates:
  Easting range: 500000.00 to 596450.15 m
  Northing range: 3318785.35 to 3418947.82 m

Example transformation:
  Lat/Lon: (30.000000°, 75.000000°)
  UTM: (500000.00 m E, 3318785.35 m N)
Coordinates transformed

UTM coordinates:
  Easting range: 500000.00 to 596450.15 m
  Northing range: 3318785.35 to 3418947.82 m

Example transformation:
  Lat/Lon: (30.000000°, 75.000000°)
  UTM: (500000.00 m E, 3318785.35 m N)


## 🏗️ Setup ANUGA Domain

In [5]:
def parse_qgis_bounding_box(csv_path, buffer_distance=-5000.0):
    """Parse QGIS bounding box from CSV with optional buffering."""
    df = pd.read_csv(csv_path)
    wkt_string = df['WKT'].iloc[0]
    polygon_geom = wkt.loads(wkt_string)
    
    if buffer_distance != 0:
        polygon_geom = polygon_geom.buffer(buffer_distance)
    
    coords_tuples = list(polygon_geom.exterior.coords)
    anuga_polygon = [list(pt) for pt in coords_tuples]
    
    if anuga_polygon[0] == anuga_polygon[-1]:
        anuga_polygon.pop()
    
    return anuga_polygon

print("\n" + "="*70)
print("CREATING ANUGA DOMAIN")
print("="*70)

# Load boundary polygon
print(f"\nLoading boundary from {config.bounding_box_csv}...")
bounding_polygon = parse_qgis_bounding_box(config.bounding_box_csv, config.buffer_distance)
print(f"Parsed {len(bounding_polygon)} vertices")

# Create mesh
print(f"\nCreating computational mesh (max triangle area: {config.maximum_triangle_area} m²)...")
num_segments = len(bounding_polygon)
tags = {'exterior': list(range(num_segments))}

start_time = time.time()
domain = anuga.create_domain_from_regions(
    bounding_polygon,
    boundary_tags=tags,
    maximum_triangle_area=config.maximum_triangle_area
)
mesh_time = time.time() - start_time

print(f"Domain created in {mesh_time:.2f}s")
print(f"  Triangles: {domain.get_number_of_triangles():,}")
print(f"  Vertices: {domain.get_number_of_nodes():,}")
print(f"  Area: {domain.get_area()/1e6:.2f} km²")

# Set coordinate reference
domain.geo_reference.set_zone(43)  # UTM Zone 43N
domain.geo_reference.set_hemisphere('northern')
print(f"Set UTM Zone 43N (Northern Hemisphere)")

# Configure flow algorithm for GPU
domain.set_flow_algorithm('DE0')
domain.set_low_froude(0)
print(f"Flow algorithm: DE0 (GPU compatible)")


CREATING ANUGA DOMAIN

Loading boundary from bounder.csv...
Parsed 4 vertices

Creating computational mesh (max triangle area: 30000 m²)...
Setting omp_num_threads to 1
Domain created in 6.74s
  Triangles: 446,030
  Vertices: 223,853
  Area: 8666.64 km²
Set UTM Zone 43N (Northern Hemisphere)
Flow algorithm: DE0 (GPU compatible)
Setting omp_num_threads to 1
Domain created in 6.74s
  Triangles: 446,030
  Vertices: 223,853
  Area: 8666.64 km²
Set UTM Zone 43N (Northern Hemisphere)
Flow algorithm: DE0 (GPU compatible)


## 🏔️ Load Topography

In [6]:
print("\n" + "="*70)
print("LOADING TOPOGRAPHY")
print("="*70)

print(f"\nLoading DEM from {config.dem_file}...")
domain.get_quantity('elevation').set_values_from_tif_file(config.dem_file)
elev = domain.get_quantity('elevation')

print(f"Elevation loaded")
print(f"  Min elevation: {elev.get_minimum_value():.2f} m")
print(f"  Max elevation: {elev.get_maximum_value():.2f} m")
print(f"  Mean elevation: {np.mean(elev.centroid_values):.2f} m")

# Set initial conditions
print(f"\nSetting initial conditions...")
domain.set_quantity('friction', config.friction_coefficient)
domain.set_quantity('stage', expression='elevation')  # Dry bed
print(f"Friction coefficient: {config.friction_coefficient}")
print(f"Initial condition: Dry bed (stage = elevation)")

# Set boundary conditions
print(f"\nSetting boundary conditions...")
Bo = anuga.Dirichlet_boundary([-10.0, 0.0, 0.0])  # Outflow
domain.set_boundary({'exterior': Bo})
print(f"Boundary: Dirichlet outflow on all exterior boundaries")

# Configure domain
domain.set_minimum_storable_height(config.minimum_storable_height)
domain.set_name(config.simulation_name)
domain.set_datadir(config.output_dir)
os.makedirs(config.output_dir, exist_ok=True)

# Enable .sww file storage
domain.set_store(True)
domain.set_store_vertices_uniquely(False)
print(f"SWW file storage enabled")


LOADING TOPOGRAPHY

Loading DEM from filled_smoothed_topography_EPSG32643.tif...
Elevation loaded
  Min elevation: 207.94 m
  Max elevation: 263.16 m
  Mean elevation: 229.64 m

Setting initial conditions...
Friction coefficient: 0.0175
Initial condition: Dry bed (stage = elevation)

Setting boundary conditions...
Boundary: Dirichlet outflow on all exterior boundaries
SWW file storage enabled


/home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


## 🌧️ Prepare Rainfall Operator with Cell-Based Assignment

In [7]:
import gc

print("\n" + "="*70)
print("PREPARING RAINFALL DATA (DISK-BACKED MEMORY-MAPPED STORAGE)")
print("="*70)

# Filter rainfall data for simulation period
sim_start_dt = config.start_date
sim_end_dt = config.end_date

print(f"\nFiltering rainfall data for simulation period...")
print(f"  Start: {sim_start_dt}")
print(f"  End: {sim_end_dt}")

mask_time = (rain_df['datetime'] >= sim_start_dt) & (rain_df['datetime'] <= sim_end_dt)
rain_sim = rain_df[mask_time].copy()

print(f"Filtered {len(rain_sim):,} records for simulation period")

# Get simulation times and convert to seconds from start
sim_times = sorted(rain_sim['datetime'].unique())
time_seconds = np.array([(t - sim_times[0]).total_seconds() for t in sim_times])

print(f"mulation timesteps: {len(sim_times)}")
print(f"  Duration: {time_seconds[-1]/3600:.1f} hours ({time_seconds[-1]/86400:.1f} days)")

# Prepare sorted coordinate arrays for efficient lookup
lat_sorted = np.array(sorted(rain_sim['lat'].unique()))
lon_sorted = np.array(sorted(rain_sim['long'].unique()))

print(f"\nRainfall grid structure:")
print(f"  Latitude points: {len(lat_sorted)}")
print(f"  Longitude points: {len(lon_sorted)}")

# Pre-compute indices for all rainfall data points (vectorized)
datetime_array = rain_sim['datetime'].values
lat_idx = np.searchsorted(lat_sorted, rain_sim['lat'].values)
lon_idx = np.searchsorted(lon_sorted, rain_sim['long'].values)
precip_array = rain_sim['precipitation'].values.astype(np.float32)

print(f"Pre-computed indices for {len(rain_sim):,} data points")

# Create sparse rain grid (reusable array, float32 for memory efficiency)
rain_grid = np.zeros((len(lat_sorted), len(lon_sorted)), dtype=np.float32)

print(f"Allocated reusable rain grid: {rain_grid.shape} ({rain_grid.nbytes/1e6:.2f} MB)")

# Create directory for memory-mapped files
rainfall_cache_dir = 'rainfall_cache'
os.makedirs(rainfall_cache_dir, exist_ok=True)

# Define memory-mapped file paths
rainfall_mmap_file = os.path.join(rainfall_cache_dir, 'rainfall_data.dat')
time_index_file = os.path.join(rainfall_cache_dir, 'time_index.npy')

print(f"\nCache directory: {rainfall_cache_dir}/")
print(f"  - rainfall_data.dat (memory-mapped binary)")
print(f"  - time_index.npy (time index)")

# CHECK IF FILES ALREADY EXIST (SKIP REGENERATION AFTER KERNEL CRASH)
if os.path.exists(rainfall_mmap_file) and os.path.exists(time_index_file):
    print(f"\nFound existing rainfall cache files!")
    print(f"  Rainfall data: {rainfall_mmap_file} ({os.path.getsize(rainfall_mmap_file)/1e9:.2f} GB)")
    print(f"  Time index: {time_index_file}")
    print(f"\n  Loading from cache (skipping regeneration)...")
    
    # Load time index
    time_seconds = np.load(time_index_file)
    n_timesteps = len(time_seconds)
    n_triangles = domain.get_number_of_triangles()
    
    # Open existing memory-mapped file in read-only mode
    rainfall_mmap = np.memmap(rainfall_mmap_file, dtype=np.float32, mode='r',
                              shape=(n_timesteps, n_triangles))
    
    print(f"  Loaded memory-mapped file:")
    print(f"    Shape: ({n_timesteps:,} timesteps, {n_triangles:,} triangles)")
    print(f"    RAM overhead: ~0 MB (OS page cache only)")
    
    # Verify data integrity
    test_rainfall = rainfall_mmap[0, :]
    print(f"\n  Data verification (first timestep):")
    print(f"    Min: {test_rainfall.min()*3600*1000:.6f} mm/hr")
    print(f"    Max: {test_rainfall.max()*3600*1000:.6f} mm/hr")
    print(f"    Mean: {test_rainfall.mean()*3600*1000:.6f} mm/hr")
    
    print(f"\n Using cached rainfall data (kernel crash recovery)")
    print(f"   To regenerate: delete 'rainfall_cache/' directory\n")
    
    # Set flag to skip generation
    SKIP_GENERATION = True
else:
    print(f"\n Cache files not found - will generate new data")
    SKIP_GENERATION = False

if not SKIP_GENERATION:
    # Get domain coordinates
    domain_x = domain.centroid_coordinates[:, 0]
    domain_y = domain.centroid_coordinates[:, 1]
    n_triangles = len(domain_x)

    # Transform domain coordinates to WGS84
    domain_lon, domain_lat = transformer.transform(domain_x, domain_y)

    # Find indices for all domain points
    lat_indices_all = np.searchsorted(lat_sorted, domain_lat)
    lon_indices_all = np.searchsorted(lon_sorted, domain_lon)

    # Clip to valid range
    lat_indices_all = np.clip(lat_indices_all, 0, len(lat_sorted) - 2)
    lon_indices_all = np.clip(lon_indices_all, 0, len(lon_sorted) - 2)

    # Pre-compute cell indices for all domain points
    lat_ll_idx = lat_indices_all
    lon_ll_idx = lon_indices_all
    lat_ur_idx = lat_indices_all + 1
    lon_ur_idx = lon_indices_all + 1

    print(f"\nRainfall grid spacing:")
    print(f"  Latitude spacing: {lat_sorted[1] - lat_sorted[0]:.4f}° (~{(lat_sorted[1] - lat_sorted[0]) * 111000:.0f} m)")
    print(f"  Longitude spacing: {lon_sorted[1] - lon_sorted[0]:.4f}° (~{(lon_sorted[1] - lon_sorted[0]) * 111000 * np.cos(np.radians(domain_lat.mean())):.0f} m)")

    print(f"\n Creating cell-based rainfall structure (DISK-BACKED)...")
    print(f"  Method: Each cell uses average of 4 corner grid points")
    print(f"  Storage: Memory-mapped file (zero RAM overhead)")
    print(f"  Transformed {n_triangles:,} domain points to lat/lon")
    print(f"  Pre-computed cell indices for all domain points")
    print(f"  Creating efficient rainfall grid lookup...")

    # Pre-allocate reusable arrays (float32 for memory efficiency)
    v_ll = np.zeros(n_triangles, dtype=np.float32)
    v_lr = np.zeros(n_triangles, dtype=np.float32)
    v_ul = np.zeros(n_triangles, dtype=np.float32)
    v_ur = np.zeros(n_triangles, dtype=np.float32)

    array_size_gb = n_triangles * 4 * 4 / 1e9  # 4 arrays * 4 bytes (float32)
    print(f"  Pre-allocated reusable arrays (saving ~{array_size_gb * len(sim_times):.2f} GB by reusing)")

    # Create memory-mapped file for rainfall data
    n_timesteps = len(sim_times)
    rainfall_mmap = np.memmap(rainfall_mmap_file, dtype=np.float32, mode='w+',
                              shape=(n_timesteps, n_triangles))

    print(f"\nMemory-mapped file created:")
    print(f"  File: {rainfall_mmap_file}")
    print(f"  Shape: ({n_timesteps:,} timesteps, {n_triangles:,} triangles)")
    print(f"  Size on disk: ~{n_timesteps * n_triangles * 4 / 1e9:.2f} GB")
    print(f"  RAM overhead: ~0 MB (OS page cache only)")

    # Process all timesteps and write to memory-mapped file
    gc_interval = 500  # Run GC every N timesteps
    for i, t in enumerate(sim_times):
        # Filter data for this time
        mask = datetime_array == t
        lat_idx_t = lat_idx[mask]
        lon_idx_t = lon_idx[mask]
        precip_t = precip_array[mask]
        
        # Create sparse rain grid (float32 for memory efficiency)
        rain_grid.fill(0.0)  # Reuse array, reset to zero
        rain_grid[lat_idx_t, lon_idx_t] = precip_t
        
        # MEMORY-CRITICAL: Use pre-allocated arrays (in-place copy operations)
        np.copyto(v_ll, rain_grid[lat_ll_idx, lon_ll_idx])
        np.copyto(v_lr, rain_grid[lat_ll_idx, lon_ur_idx])
        np.copyto(v_ul, rain_grid[lat_ur_idx, lon_ll_idx])
        np.copyto(v_ur, rain_grid[lat_ur_idx, lon_ur_idx])
        
        # Average of 4 corners and convert mm/hr to m/s, write directly to mmap
        rainfall_mmap[i, :] = (v_ll + v_lr + v_ul + v_ur) * (0.25 / 3600000.0)
        
        # MEMORY-CRITICAL: Flush to disk and GC at intervals
        if (i + 1) % gc_interval == 0:
            rainfall_mmap.flush()  # Write buffered data to disk
            gc.collect()
            print(f"  Progress: {i+1}/{len(sim_times)} timesteps | Flushed to disk + GC")
        elif (i + 1) % (gc_interval * 2) == 0:
            print(f"  Progress: {i+1}/{len(sim_times)} timesteps")

    # Final flush and garbage collection
    rainfall_mmap.flush()
    gc.collect()

    print(f"\nCreated memory-mapped rainfall file with {n_timesteps:,} timesteps")
    mmap_file_size_gb = n_timesteps * n_triangles * 4 / 1e9  # float32 = 4 bytes
    print(f"  File size on disk: ~{mmap_file_size_gb:.2f} GB")
    print(f"  RAM overhead: ~0 MB (OS manages paging)")

    # Save time index for operator lookup
    np.save(time_index_file, time_seconds)
    print(f"Saved time index to {time_index_file}")

    # Clean up temporary arrays to free memory
    del v_ll, v_lr, v_ul, v_ur, rain_grid, lat_indices_all, lon_indices_all
    del datetime_array, precip_array, mask, lat_idx_t, lon_idx_t, precip_t
    gc.collect()

    print(f"\nCleaned up temporary arrays")

    # Reopen in read-only mode for verification
    del rainfall_mmap  # Close write mode
    rainfall_mmap = np.memmap(rainfall_mmap_file, dtype=np.float32, mode='r',
                              shape=(n_timesteps, n_triangles))

# Test at first timestep (works for both cached and newly generated)
print(f"\nVerifying data integrity (reading from disk)...")
test_rainfall = rainfall_mmap[0, :]
print(f"  Min rainfall: {test_rainfall.min()*3600*1000:.6f} mm/hr")
print(f"  Max rainfall: {test_rainfall.max()*3600*1000:.6f} mm/hr")
print(f"  Mean rainfall: {test_rainfall.mean()*3600*1000:.6f} mm/hr")
print(f"  Non-zero points: {np.sum(test_rainfall > 0):,} ({np.sum(test_rainfall > 0)/len(test_rainfall)*100:.1f}%)")

print(f"\nMemory-mapped storage ready for simulation")
print(f"  Data accessible on-demand with zero RAM overhead")

# Final confirmation (works for both cached and newly generated)
print(f"\n{'='*70}")
print(f"RAINFALL DATA READY")
print(f"{'='*70}")
print(f"  Timesteps: {n_timesteps:,}")
print(f"  Domain triangles: {n_triangles:,}")
print(f"  Storage: {rainfall_mmap_file}")
print(f"  Status: {' Reused from cache' if SKIP_GENERATION else ' Newly generated'}")
print(f"{'='*70}\n")


PREPARING RAINFALL DATA (DISK-BACKED MEMORY-MAPPED STORAGE)

Filtering rainfall data for simulation period...
  Start: 2022-05-15 00:00:00
  End: 2022-09-15 23:59:59
Filtered 327,360 records for simulation period
mulation timesteps: 2976
  Duration: 2975.0 hours (124.0 days)

Rainfall grid structure:
  Latitude points: 10
  Longitude points: 11
Pre-computed indices for 327,360 data points
Allocated reusable rain grid: (10, 11) (0.00 MB)

Cache directory: rainfall_cache/
  - rainfall_data.dat (memory-mapped binary)
  - time_index.npy (time index)

Found existing rainfall cache files!
  Rainfall data: rainfall_cache/rainfall_data.dat (5.31 GB)
  Time index: rainfall_cache/time_index.npy

  Loading from cache (skipping regeneration)...
  Loaded memory-mapped file:
    Shape: (2,976 timesteps, 446,030 triangles)
    RAM overhead: ~0 MB (OS page cache only)

  Data verification (first timestep):
    Min: 0.000000 mm/hr
    Max: 0.000000 mm/hr
    Mean: 0.000000 mm/hr

 Using cached rainfal

## 🎯 Create Custom Rainfall Operator Class

In [8]:
class Spatial_Temporal_Rainfall_Operator(anuga.Rate_operator):
    """
    Custom rainfall operator that applies cell-based constant rainfall values.
    
    Uses ANUGA's Rate_operator which properly handles source terms.
    Reads rainfall data from memory-mapped file for zero RAM overhead.
    """
    
    def __init__(self, domain, rainfall_mmap_file, time_seconds, 
                 description=None, label=None, logging=False, verbose=False):
        """
        Initialize the spatial-temporal rainfall operator.
        
        Parameters:
        -----------
        domain : ANUGA Domain
            The computational domain
        rainfall_mmap_file : str
            Path to memory-mapped rainfall data file
        time_seconds : array
            Array of time values (seconds) for which rainfall data exists
        """
        self.time_seconds = time_seconds
        self.n_triangles = domain.get_number_of_triangles()
        self.n_timesteps = len(time_seconds)
        self.current_time_idx = -1
        self.call_counter = 0  # Track how many times __call__ is invoked
        
        # Open memory-mapped file in read-only mode
        self.rainfall_mmap = np.memmap(rainfall_mmap_file, dtype=np.float32, mode='r',
                                       shape=(self.n_timesteps, self.n_triangles))
        
        print(f"Memory-mapped rainfall file opened (read-only)")
        print(f"  File: {rainfall_mmap_file}")
        print(f"  Shape: ({self.n_timesteps:,} timesteps, {self.n_triangles:,} triangles)")
        print(f"  RAM overhead: ~0 MB (OS page cache only)")
        
        # Initialize with first rainfall rate (read from mmap)
        initial_rate = self.rainfall_mmap[0, :].copy()  # Copy to avoid mmap reference
        
        # Initialize Rate_operator with initial rate
        anuga.Rate_operator.__init__(self, domain, rate=initial_rate,
                        description=description, label=label,
                        logging=logging, verbose=verbose)
        
        print(f"Cell-Based Rainfall Operator initialized (DISK-BACKED)")
        print(f"  Domain triangles: {self.n_triangles:,}")
        print(f"  Time steps: {len(self.time_seconds)}")
        print(f"  Method: Memory-mapped file access (zero RAM overhead)")
    
    def __call__(self):
        """
        CRITICAL: Override __call__ to ensure rainfall is actually applied.
        
        ANUGA's Rate_operator doesn't automatically call update_rate().
        We must explicitly call update_rate() here, then let the parent
        Rate_operator.__call__() apply the rate to the domain.
        """
        self.call_counter += 1
        
        # Read current rainfall from memory-mapped file
        self.update_rate()
        
        # Now apply the rate (this is what actually adds water)
        anuga.Rate_operator.__call__(self)
        
        # Debug output for first 5 calls to confirm it's working
        if self.call_counter <= 5:
            t = self.domain.get_time()
            print(f"DEBUG Rainfall __call__() #{self.call_counter} at t={t/3600:.2f}h: "
                  f"rate min={self.rate.min()*3600*1000:.4f} mm/hr, "
                  f"max={self.rate.max()*3600*1000:.4f} mm/hr")
    
    def update_rate(self):
        """
        Update the rainfall rate based on current simulation time.
        Reads data from memory-mapped file on-demand.
        """
        t = self.domain.get_time()
        
        # Find the appropriate rainfall array index for current time
        idx = np.searchsorted(self.time_seconds, t)
        
        if idx >= len(self.time_seconds):
            idx = len(self.time_seconds) - 1
        elif idx > 0 and abs(t - self.time_seconds[idx-1]) < abs(t - self.time_seconds[idx]):
            idx = idx - 1
        
        # Read rainfall data from memory-mapped file (fast, OS-cached)
        rainfall_rate = self.rainfall_mmap[idx, :]  # Read row idx
        
        # Ensure non-negative
        rainfall_rate = np.maximum(rainfall_rate, 0.0)
        
        # Update the rate (Rate_operator will apply this)
        self.set_rate(rainfall_rate)
        
        # Update monitoring
        if idx != self.current_time_idx:
            self.current_time_idx = idx
            if self.verbose and idx % 24 == 0:  # Print every 24 hours
                print(f"  Rainfall updated at t={t/3600:.1f}h: "
                      f"min={rainfall_rate.min()*3600*1000:.3f} mm/hr, "
                      f"max={rainfall_rate.max()*3600*1000:.3f} mm/hr "
                      f"(read from disk)")
    
    def parallel_safe(self):
        """Required for ANUGA operator framework."""
        return True
    
    def statistics(self):
        """Return statistics about current rainfall."""
        if hasattr(self, 'rate') and self.rate is not None:
            if isinstance(self.rate, np.ndarray):
                rate_array = self.rate
            else:
                rate_array = np.full(self.n_triangles, self.rate)
            
            return {
                'min_rate_mm_hr': float(rate_array.min() * 3600 * 1000),
                'max_rate_mm_hr': float(rate_array.max() * 3600 * 1000),
                'mean_rate_mm_hr': float(rate_array.mean() * 3600 * 1000)
            }
        return {}
    
    def __del__(self):
        """Clean up memory-mapped file on deletion."""
        if hasattr(self, 'rainfall_mmap'):
            del self.rainfall_mmap

print("Custom Cell-Based Rainfall Operator class defined (DISK-BACKED)")


Custom Cell-Based Rainfall Operator class defined (DISK-BACKED)


## 📊 Analyze Precipitation Statistics Over Simulation Period

In [9]:
print("\n" + "="*70)
print("PRECIPITATION ANALYSIS OVER SIMULATION PERIOD")
print("="*70)

# OPTIMIZED: Read from memory-mapped file for zero RAM overhead
# Pre-allocate arrays for statistics
timestep_mins = np.zeros(n_timesteps, dtype=np.float32)
timestep_maxs = np.zeros(n_timesteps, dtype=np.float32)
timestep_means = np.zeros(n_timesteps, dtype=np.float32)
timestep_nonzero_pct = np.zeros(n_timesteps, dtype=np.float32)
timestep_hours = time_seconds / 3600.0

print(f"Analyzing {n_timesteps:,} timesteps from memory-mapped file...")
print(f"  (Reading on-demand, ~0 MB RAM overhead)")

# Vectorized computation reading from mmap (OS caches frequently accessed data)
gc_interval = 500  # Run GC every N timesteps during analysis
for i in range(n_timesteps):
    # Read rainfall data from mmap (fast, OS-cached)
    rates_m_s = rainfall_mmap[i, :]
    
    # Convert to mm/hr for analysis
    rates_mm_hr = rates_m_s * 3600 * 1000
    
    timestep_mins[i] = rates_mm_hr.min()
    timestep_maxs[i] = rates_mm_hr.max()
    timestep_means[i] = rates_mm_hr.mean()
    timestep_nonzero_pct[i] = (rates_m_s > 0).sum() / len(rates_m_s) * 100
    
    # MEMORY-CRITICAL: GC at intervals to prevent accumulation
    if (i + 1) % gc_interval == 0:
        gc.collect()

# Final GC after analysis
gc.collect()
print("Analysis complete")

# ========== OVERALL STATISTICS ==========
print(f"\n{'='*70}")
print("OVERALL PRECIPITATION STATISTICS")
print(f"{'='*70}")
print(f"Total timesteps analyzed: {n_timesteps:,}")
print(f"Duration: {timestep_hours[-1]:.1f} hours ({timestep_hours[-1]/24:.1f} days)")

print(f"\n Across All Timesteps:")
print(f"  Min rainfall (any cell, any time): {timestep_mins.min():.6f} mm/hr")
print(f"  Max rainfall (any cell, any time): {timestep_maxs.max():.6f} mm/hr")
print(f"  Mean of timestep means: {timestep_means.mean():.6f} mm/hr")
print(f"  Mean coverage (cells with rain): {timestep_nonzero_pct.mean():.2f}%")

# ========== PEAK RAINFALL EVENTS ==========
max_mean_idx = timestep_means.argmax()
max_max_idx = timestep_maxs.argmax()

print(f"\n Peak Rainfall Events:")
print(f"  Highest mean rainfall: {timestep_means[max_mean_idx]:.6f} mm/hr at t={timestep_hours[max_mean_idx]:.1f} hrs")
print(f"  Highest max rainfall: {timestep_maxs[max_max_idx]:.6f} mm/hr at t={timestep_hours[max_max_idx]:.1f} hrs")

# ========== TIME SERIES SUMMARY ==========
print(f"\n Temporal Patterns:")
# Identify dry periods (mean rainfall < 0.01 mm/hr)
dry_mask = timestep_means < 0.01
wet_mask = ~dry_mask
n_dry = dry_mask.sum()
n_wet = wet_mask.sum()

print(f"  Dry timesteps (< 0.01 mm/hr mean): {n_dry:,} ({n_dry/n_timesteps*100:.1f}%)")
print(f"  Wet timesteps (≥ 0.01 mm/hr mean): {n_wet:,} ({n_wet/n_timesteps*100:.1f}%)")

if n_wet > 0:
    print(f"\n💧 Statistics for Wet Timesteps Only:")
    print(f"  Mean rainfall: {timestep_means[wet_mask].mean():.6f} mm/hr")
    print(f"  Max rainfall: {timestep_means[wet_mask].max():.6f} mm/hr")
    print(f"  Mean coverage: {timestep_nonzero_pct[wet_mask].mean():.2f}%")

# ========== SAMPLE TIMESTEPS ==========
# Show detailed info for first, middle, and last timesteps
sample_indices = [0, n_timesteps // 2, -1]
print(f"\nSample Timestep Details:")
for idx in sample_indices:
    if idx == -1:
        idx = n_timesteps - 1
    t_hr = timestep_hours[idx]
    print(f"  t={t_hr:7.1f}h: mean={timestep_means[idx]:8.6f} mm/hr, "
          f"max={timestep_maxs[idx]:8.6f} mm/hr, "
          f"coverage={timestep_nonzero_pct[idx]:5.1f}%")

print(f"\n{'='*70}\n")

# Clean up analysis arrays
del timestep_mins, timestep_maxs, timestep_means, timestep_nonzero_pct
del timestep_hours
gc.collect()
print("Analysis arrays cleaned up")



PRECIPITATION ANALYSIS OVER SIMULATION PERIOD
Analyzing 2,976 timesteps from memory-mapped file...
  (Reading on-demand, ~0 MB RAM overhead)
Analysis complete

OVERALL PRECIPITATION STATISTICS
Total timesteps analyzed: 2,976
Duration: 2975.0 hours (124.0 days)

 Across All Timesteps:
  Min rainfall (any cell, any time): 0.000000 mm/hr
  Max rainfall (any cell, any time): 14.346251 mm/hr
  Mean of timestep means: 0.173512 mm/hr
  Mean coverage (cells with rain): 20.50%

 Peak Rainfall Events:
  Highest mean rainfall: 14.346252 mm/hr at t=1604.0 hrs
  Highest max rainfall: 14.346251 mm/hr at t=1604.0 hrs

 Temporal Patterns:
  Dry timesteps (< 0.01 mm/hr mean): 2,378 (79.9%)
  Wet timesteps (≥ 0.01 mm/hr mean): 598 (20.1%)

💧 Statistics for Wet Timesteps Only:
  Mean rainfall: 0.863392 mm/hr
  Max rainfall: 14.346252 mm/hr
  Mean coverage: 100.00%

Sample Timestep Details:
  t=    0.0h: mean=0.000000 mm/hr, max=0.000000 mm/hr, coverage=  0.0%
  t= 1488.0h: mean=0.000000 mm/hr, max=0.000

## ⚡ Add Rainfall Operator to Domain

In [10]:
print("\n" + "="*70)
print("ADDING RAINFALL OPERATOR TO DOMAIN")
print("="*70)

# Create and add the cell-based rainfall operator (DISK-BACKED)
rainfall_op = Spatial_Temporal_Rainfall_Operator(
    domain,
    rainfall_mmap_file=rainfall_mmap_file,
    time_seconds=time_seconds,
    description="IMERG hourly cell-based rainfall (memory-mapped)",
    label="IMERG_rainfall",
    verbose=True
)

print("\nCell-based rainfall operator added to domain (DISK-BACKED)")
print("  Each domain triangle receives constant rainfall based on")
print("  the average of 4 surrounding IMERG grid points")
print("  Data stored on disk, accessed on-demand via memory mapping")



ADDING RAINFALL OPERATOR TO DOMAIN
Memory-mapped rainfall file opened (read-only)
  File: rainfall_cache/rainfall_data.dat
  Shape: (2,976 timesteps, 446,030 triangles)
  RAM overhead: ~0 MB (OS page cache only)
Cell-Based Rainfall Operator initialized (DISK-BACKED)
  Domain triangles: 446,030
  Time steps: 2976
  Method: Memory-mapped file access (zero RAM overhead)

Cell-based rainfall operator added to domain (DISK-BACKED)
  Each domain triangle receives constant rainfall based on
  the average of 4 surrounding IMERG grid points
  Data stored on disk, accessed on-demand via memory mapping


## 🚀 Enable GPU Acceleration

In [11]:
print("\n" + "="*70)
print("GPU ACCELERATION SETUP")
print("="*70)

if config.use_gpu and GPU_AVAILABLE:
    try:
        # Test GPU
        import cupy as cp
        print("\nTesting GPU accessibility...")
        test_array = cp.random.random((1000, 1000))
        test_result = cp.sum(test_array)
        print(f"GPU computation test passed")
        
        # Enable GPU mode
        print("\nEnabling GPU mode...")
        domain.set_multiprocessor_mode(2)  # 2 = GPU mode
        
        if domain.gpu_interface is not None:
            print("GPU acceleration ENABLED!")
            print(f"  GPU interface: {type(domain.gpu_interface).__name__}")
            print(f"  Multiprocessor mode: {domain.get_multiprocessor_mode()}")
            
            mempool = cp.get_default_memory_pool()
            print(f"\n  GPU Memory:")
            print(f"    Used: {mempool.used_bytes()/1e9:.3f} GB")
            print(f"    Total: {mempool.total_bytes()/1e9:.3f} GB")
            
            print("\n  💡 TIP: Monitor GPU with 'nvidia-smi -l 1' in terminal")
        else:
            raise Exception("GPU interface not created")
            
    except Exception as e:
        print(f"\n GPU setup failed: {e}")
        print("  Falling back to CPU mode...")
        
        import multiprocessing
        num_threads = multiprocessing.cpu_count()
        domain.set_omp_num_threads(num_threads)
        print(f"  CPU mode: {num_threads} OpenMP threads")
        config.use_gpu = False
else:
    import multiprocessing
    num_threads = multiprocessing.cpu_count()
    domain.set_omp_num_threads(num_threads)
    print(f"CPU mode: {num_threads} OpenMP threads")

print("\n" + "="*70)
print("DOMAIN READY FOR SIMULATION")
print("="*70)
print(f"Triangles: {domain.get_number_of_triangles():,}")
print(f"Area: {domain.get_area()/1e6:.2f} km²")
print(f"Mode: {' GPU (CUDA/CuPy)' if config.use_gpu else ' CPU (OpenMP)'}")
print("="*70)


GPU ACCELERATION SETUP

Testing GPU accessibility...
GPU computation test passed

Enabling GPU mode...
GPU computation test passed

Enabling GPU mode...
GPU acceleration ENABLED!
  GPU interface: GPU_interface
  Multiprocessor mode: 2

  GPU Memory:
    Used: 0.326 GB
    Total: 0.326 GB

  💡 TIP: Monitor GPU with 'nvidia-smi -l 1' in terminal

DOMAIN READY FOR SIMULATION
Triangles: 446,030
Area: 8666.64 km²
Mode:  GPU (CUDA/CuPy)
GPU acceleration ENABLED!
  GPU interface: GPU_interface
  Multiprocessor mode: 2

  GPU Memory:
    Used: 0.326 GB
    Total: 0.326 GB

  💡 TIP: Monitor GPU with 'nvidia-smi -l 1' in terminal

DOMAIN READY FOR SIMULATION
Triangles: 446,030
Area: 8666.64 km²
Mode:  GPU (CUDA/CuPy)


## 📈 Setup Monitoring and Checkpointing

In [12]:
# Monitoring arrays
time_series = []
max_depth_series = []
mean_depth_series = []
total_volume_series = []
timestep_series = []

# Checkpointing setup
checkpoint_counter = 0
last_checkpoint_time = 0

def save_checkpoint():
    """Save simulation state to checkpoint file."""
    global checkpoint_counter
    
    checkpoint_file = os.path.join(config.output_dir, f'checkpoint_{checkpoint_counter:03d}.pkl')
    
    checkpoint_data = {
        'time': domain.get_time(),
        'quantities': {
            'stage': domain.get_quantity('stage').centroid_values.copy(),
            'xmomentum': domain.get_quantity('xmomentum').centroid_values.copy(),
            'ymomentum': domain.get_quantity('ymomentum').centroid_values.copy(),
        },
        'timeseries': {
            'time': np.array(time_series),
            'max_depth': np.array(max_depth_series),
            'mean_depth': np.array(mean_depth_series),
            'total_volume': np.array(total_volume_series),
            'timestep': np.array(timestep_series)
        }
    }
    
    import pickle
    with open(checkpoint_file, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    
    print(f"\n  Checkpoint saved: {checkpoint_file}")
    checkpoint_counter += 1

print("Monitoring and checkpointing configured")

Monitoring and checkpointing configured


## 🎬 Run Simulation

In [ ]:
print("\n" + "="*70)
print("STARTING SIMULATION")
print("="*70)

# Simulation parameters
yieldstep = config.output_interval_hours * 3600  # Convert to seconds
finaltime = (config.end_date - config.start_date).total_seconds()
checkpoint_interval = config.checkpoint_interval_hours * 3600  # Convert to seconds

print(f"\nSimulation parameters:")
print(f"  Duration: {finaltime/3600:.1f} hours ({finaltime/86400:.1f} days)")
print(f"  Output interval: {yieldstep/3600:.1f} hours")
print(f"  Checkpoint interval: {checkpoint_interval/3600:.1f} hours")

# Start simulation
print(f"\n{'='*70}")
print("SIMULATION RUNNING...")
print(f"{'='*70}\n")

sim_start = time.time()
step_count = 0

try:
    # Pre-compute elevation and areas (they don't change)
    elevation = domain.get_quantity('elevation').centroid_values
    areas = domain.areas
    
    for t in domain.evolve(yieldstep=yieldstep, finaltime=finaltime):
        # Get quantities (stage is the only changing quantity)
        stage = domain.get_quantity('stage').centroid_values
        
        # Vectorized depth calculation
        depth = np.maximum(stage - elevation, 0.0)
        
        # Compute statistics (fully vectorized)
        max_depth = depth.max()
        wet_mask = depth > 0.001
        mean_depth = depth[wet_mask].mean() if wet_mask.any() else 0.0
        total_volume = (depth * areas).sum()
        current_timestep = domain.get_timestep()
        
        # Store timeseries
        time_series.append(t)
        max_depth_series.append(max_depth)
        mean_depth_series.append(mean_depth)
        total_volume_series.append(total_volume)
        timestep_series.append(current_timestep)
        
        # Progress report
        step_count += 1
        elapsed = time.time() - sim_start
        progress = (t / finaltime) * 100
        eta = (elapsed / t) * (finaltime - t) if t > 0 else 0
        
        print(f"Time: {t/3600:.1f}h / {finaltime/3600:.1f}h ({progress:.1f}%) | "
              f"Max depth: {max_depth:.5f}m | "
              f"Mean depth: {mean_depth:.5f}m | "
              f"Volume: {total_volume:.4f}m³ | "
              f"dt: {current_timestep:.5f}s | "
              f"ETA: {eta/60:.0f}min")
        
        # Checkpoint
        if config.save_checkpoints and (t - last_checkpoint_time) >= checkpoint_interval:
            save_checkpoint()
            last_checkpoint_time = t
            
            # Save timeseries data
            np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
                    time=np.array(time_series),
                    max_depth=np.array(max_depth_series),
                    mean_depth=np.array(mean_depth_series),
                    total_volume=np.array(total_volume_series),
                    timestep=np.array(timestep_series))
    
    # Final save
    save_checkpoint()
    np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
            time=np.array(time_series),
            max_depth=np.array(max_depth_series),
            mean_depth=np.array(mean_depth_series),
            total_volume=np.array(total_volume_series),
            timestep=np.array(timestep_series))
    
    total_time = time.time() - sim_start
    
    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE!")
    print(f"{'='*70}")
    print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
    print(f"Time steps: {step_count}")
    print(f"Average time per step: {total_time/step_count:.2f} seconds")
    print(f"\nResults saved to: {config.output_dir}/")
    print(f"  - {config.simulation_name}.sww (full spatial results)")
    print(f"  - timeseries_data.npz (time series data)")
    print(f"  - checkpoint_*.pkl (checkpoints)")
    
except KeyboardInterrupt:
    print("\n\n⚠ SIMULATION INTERRUPTED BY USER")
    print("Saving current state...")
    save_checkpoint()
    np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
            time=np.array(time_series),
            max_depth=np.array(max_depth_series),
            mean_depth=np.array(mean_depth_series),
            total_volume=np.array(total_volume_series),
            timestep=np.array(timestep_series))
    print("State saved. You can analyze partial results.")

except Exception as e:
    print(f"\n\n ERROR: {e}")
    import traceback
    traceback.print_exc()
    print("\nAttempting to save current state...")
    try:
        save_checkpoint()
    except:
        print("Failed to save checkpoint")


STARTING SIMULATION

Simulation parameters:
  Duration: 2976.0 hours (124.0 days)
  Output interval: 24.0 hours
  Checkpoint interval: 360.0 hours

SIMULATION RUNNING...

Time: 0.0h / 2976.0h (0.0%) | Max depth: 0.00000m | Mean depth: 0.00000m | Volume: 0.0000m³ | dt: 0.00000s | ETA: 0min
  Rainfall updated at t=0.0h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
DEBUG Rainfall __call__() #1 at t=0.00h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
DEBUG Rainfall __call__() #2 at t=0.28h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
DEBUG Rainfall __call__() #3 at t=0.56h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
Time: 0.0h / 2976.0h (0.0%) | Max depth: 0.00000m | Mean depth: 0.00000m | Volume: 0.0000m³ | dt: 0.00000s | ETA: 0min
  Rainfall updated at t=0.0h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
DEBUG Rainfall __call__() #1 at t=0.00h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
DEBUG Rainfall __call__() #2 at t=0.28h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
DEBUG Rainfall __call__()

/home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/anuga/shallow_water/shallow_water_domain.py:2186: UserWarning: 446030 negative cells being set to zero depth, possible loss of conservation. 
Consider using domain.report_water_volume_statistics() to check the extent of the problem
  warnings.warn(msg)


DEBUG Rainfall __call__() #4 at t=0.83h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
DEBUG Rainfall __call__() #5 at t=1.11h: rate min=0.0000 mm/hr, max=0.0000 mm/hr
  Rainfall updated at t=23.6h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
Time: 24.0h / 2976.0h (0.8%) | Max depth: 0.00000m | Mean depth: 0.00000m | Volume: 0.0000m³ | dt: 400.00000s | ETA: 11min
  Rainfall updated at t=23.6h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
Time: 24.0h / 2976.0h (0.8%) | Max depth: 0.00000m | Mean depth: 0.00000m | Volume: 0.0000m³ | dt: 400.00000s | ETA: 11min
  Rainfall updated at t=47.5h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
  Rainfall updated at t=47.5h: min=0.000 mm/hr, max=0.000 mm/hr (read from disk)
Time: 48.0h / 2976.0h (1.6%) | Max depth: 0.03029m | Mean depth: 0.00201m | Volume: 4128872.1114m³ | dt: 4.44940s | ETA: 317min
Time: 48.0h / 2976.0h (1.6%) | Max depth: 0.03029m | Mean depth: 0.00201m | Volume: 4128872.1114m³ | dt: 4.44940s | ETA: 317min


## 📊 Visualize Results

In [ ]:
print("\n" + "="*70)
print("VISUALIZING RESULTS")
print("="*70)

# Load timeseries data
data = np.load(os.path.join(config.output_dir, 'timeseries_data.npz'))
time_hrs = data['time'] / 3600.0
max_depth = data['max_depth']
mean_depth = data['mean_depth']
total_volume = data['total_volume']
timestep = data['timestep']

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Max depth
ax1 = axes[0, 0]
ax1.plot(time_hrs, max_depth, 'b-', linewidth=2)
ax1.set_xlabel('Time (hours)', fontsize=12)
ax1.set_ylabel('Maximum Depth (m)', fontsize=12)
ax1.set_title('Maximum Water Depth Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Mean depth
ax2 = axes[0, 1]
ax2.plot(time_hrs, mean_depth, 'g-', linewidth=2)
ax2.set_xlabel('Time (hours)', fontsize=12)
ax2.set_ylabel('Mean Depth (m)', fontsize=12)
ax2.set_title('Mean Water Depth Over Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Total volume
ax3 = axes[1, 0]
ax3.plot(time_hrs, total_volume, 'r-', linewidth=2)
ax3.set_xlabel('Time (hours)', fontsize=12)
ax3.set_ylabel('Total Volume (m³)', fontsize=12)
ax3.set_title('Total Water Volume Over Time', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Timestep
ax4 = axes[1, 1]
ax4.plot(time_hrs, timestep, 'purple', linewidth=2)
ax4.set_xlabel('Time (hours)', fontsize=12)
ax4.set_ylabel('Timestep (s)', fontsize=12)
ax4.set_title('Adaptive Timestep Over Time', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
output_file = os.path.join(config.output_dir, 'simulation_results.png')
plt.savefig(output_file, dpi=150, bbox_inches='tight')
print(f"\nsults plot saved: {output_file}")
plt.show()

# Print summary statistics
print(f"\n{'='*70}")
print("SIMULATION SUMMARY")
print(f"{'='*70}")
print(f"Maximum depth reached: {np.max(max_depth):.3f} m")
print(f"Maximum volume: {np.max(total_volume):.1f} m³")
print(f"Mean depth (overall): {np.mean(mean_depth):.3f} m")
print(f"Average timestep: {np.mean(timestep):.3f} s")
print(f"Min timestep: {np.min(timestep):.3f} s")
print(f"Max timestep: {np.max(timestep):.3f} s")
print(f"{'='*70}")

## 🎥 Create Animation (Optional)

In [ ]:
def create_animation(quantity='depth', frames=100, fps=10):
    """
    Create animation from SWW file.
    
    Parameters:
    -----------
    quantity : str
        Quantity to visualize ('depth', 'stage', 'speed', etc.)
    frames : int
        Number of frames to extract
    fps : int
        Frames per second
    """
    from matplotlib.animation import FuncAnimation, PillowWriter
    from matplotlib.tri import Triangulation
    from anuga.file.netcdf import NetCDFFile
    
    print(f"\nCreating animation for {quantity}...")
    
    sww_file = os.path.join(config.output_dir, f'{config.simulation_name}.sww')
    if not os.path.exists(sww_file):
        print(f" SWW file not found: {sww_file}")
        return
    
    # Open SWW file
    fid = NetCDFFile(sww_file, 'r')
    
    # Get mesh
    x = fid.variables['x'][:]
    y = fid.variables['y'][:]
    volumes = fid.variables['volumes'][:]
    time = fid.variables['time'][:]
    
    tri = Triangulation(x, y, volumes)
    
    # Get data (all operations vectorized)
    if quantity == 'depth':
        stage = fid.variables['stage'][:]
        elevation = fid.variables['elevation'][:]
        data = np.maximum(stage - elevation, 0.0)  # Vectorized
        label = 'Water Depth (m)'
        cmap = 'Blues'
    elif quantity == 'stage':
        data = fid.variables['stage'][:]
        label = 'Water Stage (m)'
        cmap = 'viridis'
    elif quantity == 'speed':
        xmom = fid.variables['xmomentum'][:]
        ymom = fid.variables['ymomentum'][:]
        stage = fid.variables['stage'][:]
        elevation = fid.variables['elevation'][:]
        depth = np.maximum(stage - elevation, 0.001)
        # Vectorized velocity and speed calculation
        u = np.divide(xmom, depth, where=depth>0.001, out=np.zeros_like(xmom))
        v = np.divide(ymom, depth, where=depth>0.001, out=np.zeros_like(ymom))
        data = np.hypot(u, v)  # Faster than sqrt(u**2 + v**2)
        label = 'Flow Speed (m/s)'
        cmap = 'Reds'
    
    fid.close()
    
    # Select frames
    n_times = len(time)
    frame_indices = np.linspace(0, n_times-1, min(frames, n_times), dtype=int)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Initial plot
    vmin = np.percentile(data, 1)
    vmax = np.percentile(data, 99)
    
    tpc = ax.tripcolor(tri, data[0], cmap=cmap, vmin=vmin, vmax=vmax, shading='flat')
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)', fontsize=12)
    ax.set_ylabel('Northing (m)', fontsize=12)
    title = ax.set_title(f'{label} at t=0.0 hours', fontsize=14, fontweight='bold')
    cbar = plt.colorbar(tpc, ax=ax, label=label)
    
    def update(frame):
        idx = frame_indices[frame]
        tpc.set_array(data[idx])
        title.set_text(f'{label} at t={time[idx]/3600:.1f} hours')
        return tpc, title
    
    anim = FuncAnimation(fig, update, frames=len(frame_indices), interval=1000/fps, blit=False)
    
    # Save
    output_file = os.path.join(config.output_dir, f'{quantity}_animation.gif')
    writer = PillowWriter(fps=fps)
    anim.save(output_file, writer=writer)
    
    print(f"Animation saved: {output_file}")
    plt.close()

# Create animations
print("\n" + "="*70)
print("CREATING ANIMATIONS")
print("="*70)

try:
    create_animation(quantity='depth', frames=50, fps=5)
    create_animation(quantity='speed', frames=50, fps=5)
    print("\nAnimations created successfully!")
except Exception as e:
    print(f"\n Animation creation failed: {e}")
    print("  (This is optional - main results are still available)")

##  Simulation Complete!

Your rainfall simulation is complete. The results include:

1. **SWW file**: Full spatial-temporal data for detailed analysis
2. **Timeseries data**: NPZ file with depth, volume, and timestep data
3. **Plots**: PNG images showing simulation progress
4. **Animations**: GIF files showing flow evolution
5. **Checkpoints**: PKL files for recovery if needed

All files are saved in the `outputs/` directory.